In [17]:
import os
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

In [18]:
VIEWS = [
    "frontal1",
    "frontal2",
    "frontal3",
    "frontal4",
    "back",
    "lateralleft",
    "lateralright",
    "selfie",
    "handswide"
]

In [19]:
class MultiViewChildDataset(Dataset):
    def __init__(self, root_dir, transform=None, max_samples=100):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = {}

        # Build sample map from filesystem
        for view in VIEWS:
            view_path = os.path.join(root_dir, view)
            if not os.path.isdir(view_path):
                continue

            for fname in os.listdir(view_path):
                if not fname.lower().endswith((".jpg", ".png")):
                    continue

                key = os.path.splitext(fname)[0]

                if key not in self.samples:
                    self.samples[key] = {}

                self.samples[key][view] = os.path.join(view_path, fname)

        self.keys = list(self.samples.keys())[:max_samples]

        if len(self.keys) == 0:
            raise RuntimeError("No images found — check root_dir.")

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        key = self.keys[idx]

        images = []
        mask = []

        for view in VIEWS:
            if view in self.samples[key]:
                img = Image.open(self.samples[key][view]).convert("RGB")
                if self.transform:
                    img = self.transform(img)
                images.append(img)
                mask.append(1.0)
            else:
                images.append(torch.zeros(3, 224, 224))
                mask.append(0.0)

        images = torch.stack(images)
        mask = torch.tensor(mask)

        return images, mask, key


In [20]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [21]:
class MultiViewResNet18(nn.Module):
    def __init__(self):
        super().__init__()

        # Explicit ResNet-18
        base_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.cnn = nn.Sequential(*list(base_model.children())[:-1])  # remove FC

        self.height_head = nn.Linear(512, 1)
        self.gender_head = nn.Linear(512, 2)

    def forward(self, images, mask):
        """
        images: (B, V, 3, 224, 224)
        mask:   (B, V)
        """

        B, V, C, H, W = images.shape
        images = images.view(B * V, C, H, W)

        feats = self.cnn(images)          # (B*V, 512, 1, 1)
        feats = feats.view(B, V, 512)     # (B, V, 512)

        mask = mask.unsqueeze(-1)          # (B, V, 1)

        # Mask-aware average pooling
        pooled = (feats * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)

        height = self.height_head(pooled).squeeze(1)
        gender = self.gender_head(pooled)

        return height, gender


In [22]:
dataset = MultiViewChildDataset(
    root_dir="../Anthrovision Dataset/fulldataset",
    transform=transform,
    max_samples=100
)

loader = DataLoader(dataset, batch_size=8, shuffle=False)

In [23]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = MultiViewResNet18().to(device)
model.eval()


MultiViewResNet18(
  (cnn): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats

In [24]:
results = []

with torch.no_grad():
    for images, mask, keys in loader:
        images = images.to(device)
        mask = mask.to(device)

        pred_height, pred_gender = model(images, mask)

        for i in range(len(keys)):
            results.append({
                "child_id": keys[i],
                "pred_height_cm": pred_height[i].item(),
                "pred_gender": int(pred_gender[i].argmax()),
                "num_views_used": int(mask[i].sum().item())
            })


In [25]:
import pandas as pd

df = pd.DataFrame(results)
df.to_csv("multiview_predictions.csv", index=False)
df.head()

,child_id,pred_height_cm,pred_gender,num_views_used
0,IMG_20221103_112119_1_frontal1,-0.394797,1,1
1,IMG_20221103_112803_2_frontal1,-0.741062,0,1
2,IMG_20221103_113616_3_frontal1,-0.451225,1,1
3,IMG_20221103_120128_4_frontal1,-0.400750,0,1
4,IMG_20221103_122212_5_frontal1,-0.477996,1,1
